In [ ]:
%pip install -U "gradio[mcp]" groq requests
  

In [1]:
import sys
from pathlib import Path

python_em_uso = Path(sys.executable)
print("Kernel Python:", python_em_uso)
print("Versão:", sys.version.split()[0])

esperado = Path(r"C:\Users\eugen\Documents\.venv-cuda312\Scripts\python.exe")
if python_em_uso.resolve() != esperado.resolve():
    print("ATENÇÃO: selecione o kernel Python (kiro) no Jupyter antes de continuar.")
else:
    print("Ambiente kiro confirmado.")


Kernel Python: c:\Users\eugen\Documents\.venv-cuda312\Scripts\python.exe
Versão: 3.12.10
Ambiente kiro confirmado.


In [1]:
import os
from getpass import getpass
from groq import Groq

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Digite sua nova GROQ_API_KEY: ").strip()

if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError("Defina a variável de ambiente GROQ_API_KEY!")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Cliente Groq inicializado.")

Cliente Groq inicializado.


In [2]:
import requests
import gradio as gr
from groq import GroqError

HN_SEARCH_URL = "https://hn.algolia.com/api/v1/search_by_date"


def resumo_noticias(
    tema: str,
    n: int = 5,
    modelo: str = "openai/gpt-oss-120b",
) -> str:
    """Busca as n manchetes mais recentes sobre um tema no Hacker News e as resume.

    Args:
        tema: Assunto pesquisado no Hacker News.
        n: Quantidade de manchetes (3 a 10).
        modelo: Identificador do modelo de linguagem disponível na Groq.

    Returns:
        Um parágrafo em português brasileiro resumindo apenas as manchetes.
    """
    tema = tema.strip()
    if not tema:
        return "Informe um tema para pesquisar."

    n = max(3, min(int(n), 10))

    try:
        resposta = requests.get(
            HN_SEARCH_URL,
            params={"query": tema, "tags": "story", "hitsPerPage": n},
            timeout=20,
        )
        resposta.raise_for_status()
        hits = resposta.json().get("hits", [])
    except (requests.RequestException, ValueError) as exc:
        return f"Erro ao consultar o Hacker News: {type(exc).__name__}: {exc}"

    titulos = [
        titulo
        for hit in hits
        if (titulo := (hit.get("title") or "").strip())
    ]
    if not titulos:
        return f"Nenhuma manchete encontrada sobre '{tema}'."

    lista = "\n".join(f"- {titulo}" for titulo in titulos)
    prompt = (
        f"Tema: {tema}\n\n"
        f"Manchetes recentes do Hacker News:\n{lista}\n\n"
        "Produza um resumo curto, em um único parágrafo, em português brasileiro. "
        "Use apenas as informações contidas nas manchetes; não invente detalhes "
        "e não afirme ter lido os artigos completos."
    )

    try:
        resposta_llm = client.chat.completions.create(
            model=modelo,
            messages=[
                {
                    "role": "system",
                    "content": "Você resume manchetes de tecnologia com precisão e concisão.",
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.3,
        )
    except GroqError as exc:
        return f"Erro na API da Groq: {type(exc).__name__}: {exc}"

    resumo = (resposta_llm.choices[0].message.content or "").strip()
    return resumo or "A Groq não retornou conteúdo para o resumo."

In [4]:
demo = gr.Interface(
    fn=resumo_noticias,
    inputs=[
        gr.Textbox(label="Tema", placeholder="Ex.: AI, Rust, Serverless"),
        gr.Slider(minimum=3, maximum=10, value=5, step=1, label="Quantidade de manchetes"),
        gr.Dropdown(
            choices=["openai/gpt-oss-120b", "openai/gpt-oss-20b"],
            value="openai/gpt-oss-120b",
            label="Modelo Groq",
        ),
    ],
    outputs=gr.Textbox(label="Resumo", lines=6),
    title="Resumo de notícias do Hacker News — Groq + MCP",
    description=(
        "Pesquise as manchetes mais recentes sobre um tema e receba um resumo "
        "curto gerado pela Groq. A função também é exposta como ferramenta MCP."
    ),
    api_name="resumo_noticias",
)

In [5]:
{
  "mcpServers": {
    "hacker-news-kiro": {
      "url": "http://127.0.0.1:7860/gradio_api/mcp/"
    }
  }
}

{'mcpServers': {'hacker-news-kiro': {'url': 'http://127.0.0.1:7860/gradio_api/mcp/'}}}

In [8]:
tema = "Inteligência Artificial"

resumo = resumo_noticias(
    tema=tema,
    n=5,
    modelo="openai/gpt-oss-120b"
)

print(f"Resumo das notícias sobre {tema}:\n")
print(resumo)

Resumo das notícias sobre Inteligência Artificial:

As manchetes recentes do Hacker News abordam diferentes aspectos da inteligência artificial: há um guia completo sobre a proteção de patentes de IA no Brasil; o serviço “Render IA” oferece renders de arquitetura usando IA; surge a promessa de uma “inteligência artificial que funciona” (possivelmente ainda em avaliação); são analisados viéses e questões de IA na América Latina; e, por fim, uma reflexão em latim sugere que a inteligência artificial não supera o que a natureza já oferece.


In [9]:
from IPython.display import display, Markdown

display(Markdown(f"### Notícias sobre {tema}\n\n{resumo}"))

### Notícias sobre Inteligência Artificial

As manchetes recentes do Hacker News abordam diferentes aspectos da inteligência artificial: há um guia completo sobre a proteção de patentes de IA no Brasil; o serviço “Render IA” oferece renders de arquitetura usando IA; surge a promessa de uma “inteligência artificial que funciona” (possivelmente ainda em avaliação); são analisados viéses e questões de IA na América Latina; e, por fim, uma reflexão em latim sugere que a inteligência artificial não supera o que a natureza já oferece.